# Figure: rotational invariance with a distance-based classifier

Paper section: *Structural Properties of DIWECT Embeddings*, paragraph *Rotational Invariance with a Distance Based Classifier*

In general, a distance-based classifier fixes its metric in advance. Thus, it cannot learn to ignore rotation, and therefore the representation itself must be invariant to rotation. We classify using a Gaussian-kernel SVM ($C = 10$) and with a one-nearest neighbor (1-NN) classifier using the same squared distance.

| bar | representation | role |
|---|---|---|
| mass | total pixel weight (identical for every class) | control |
| pixels | raw pixels | not invariant |
| wect | WECT, 64 directions × 48 heights | not invariant |
| wect pca | WECT projected by PCA (fit on train) to the DIWECT's width | dimension control |
| wect rand | WECT, Gaussian random projection to 48 dims | dimension control |
| diwect | the DIWECT (48 heights) | exactly invariant |
| wect +align | WECT of the image rotated back by the true angle | oracle |

This notebook uses EAS2D generated on the fly.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.random_projection import GaussianRandomProjection
from sklearn.svm import SVC

import common as C

C.use_paper_style()
FOLDER = "2-dist-classifiers-synthetic"

SIZE, AREA = 64, 380
NUM_DIRS, NUM_HEIGHTS = 64, 48
N_TRAIN, N_TEST = 6, 40      # per class
N_SEEDS = 6
SVM_C, K_NEIGHBORS = 10.0, 1

LADDER = [   # (key, kind, role)
    ("mass", "feature", "control"),
    ("pixels", "feature", "baseline"),
    ("wect", "feature", "baseline"),
    ("wect_pca", "projector", "dim control"),
    ("wect_rand", "projector", "dim control"),
    ("diwect", "feature", "invariant"),
    ("wect_align", "feature", "oracle"),
]
ORDER = [m[0] for m in LADDER]
KIND = {k: v for k, v, _ in LADDER}
ROLE = {k: r for k, _, r in LADDER}
TICKS = {"mass": "Mass", "pixels": "Pixels", "wect": "WECT", "wect_pca": "WECT\nPCA",
         "wect_rand": "WECT\nrand. proj.", "diwect": "DIWECT", "wect_align": "WECT\n+ align"}

shapes = C.make_shape_library(size=SIZE, area=AREA)
transform = C.WECTTransform(num_dirs=NUM_DIRS, num_heights=NUM_HEIGHTS)
N_CLASSES = len(shapes)
print(f"{N_CLASSES} classes, {N_TRAIN} train / {N_TEST} test per class, {N_SEEDS} seeds, "
      f"chance = {1 / N_CLASSES:.2f}")

5 classes, 6 train / 40 test per class, 6 seeds, chance = 0.20


## Dataset and representations

In [2]:
def make_dataset(n_per_class, seed):
    rng = np.random.default_rng(seed)
    images, labels, angles = [], [], []
    for label, img in enumerate(shapes.values()):
        for _ in range(n_per_class):
            angle = rng.uniform(0.0, 360.0)
            images.append(C.rotate_image(img, angle).astype(np.float32))
            labels.append(label)
            angles.append(angle)
    return images, np.asarray(labels), np.asarray(angles)


def wect_stack(images):
    return np.stack([transform.wect(im) for im in images])   # (N, D, H)


def representations(images, angles):
    W = wect_stack(images)
    aligned = wect_stack([C.rotate_image(im, -a) for im, a in zip(images, angles)])
    return {
        "mass": np.array([[im.sum()] for im in images]),
        "pixels": np.stack([im.ravel() for im in images]),
        "wect": W.reshape(len(W), -1),
        "diwect": np.stack([transform.diwect(im) for im in images]),   # exact (src/diwect.py)
        "wect_align": aligned.reshape(len(W), -1),
    }

## Evaluation

Each representation is reduced to squared Euclidean distances: the SVM uses the Gaussian kernel $\exp(-\gamma d^2)$ with $\gamma$ from the median heuristic on the training distances, and 1-NN uses $d$ directly. The dimension controls are fit on the training set only.

In [3]:
def evaluate(tr, te, y_tr, y_te):
    d2_tr, d2_te = cdist(tr, tr, "sqeuclidean"), cdist(te, tr, "sqeuclidean")
    g = C.median_gamma(d2_tr)
    svm = SVC(kernel="precomputed", C=SVM_C).fit(np.exp(-g * d2_tr), y_tr)
    knn = KNeighborsClassifier(n_neighbors=K_NEIGHBORS, metric="precomputed").fit(np.sqrt(d2_tr), y_tr)
    return (accuracy_score(y_te, svm.predict(np.exp(-g * d2_te))),
            accuracy_score(y_te, knn.predict(np.sqrt(d2_te))))


def run_seed(seed):
    tr_im, y_tr, a_tr = make_dataset(N_TRAIN, seed)
    te_im, y_te, a_te = make_dataset(N_TEST, 1000 + seed)
    F_tr, F_te = representations(tr_im, a_tr), representations(te_im, a_te)
    projectors = {"wect_pca": PCA(n_components=min(NUM_HEIGHTS, len(y_tr) - 1)),
                  "wect_rand": GaussianRandomProjection(n_components=NUM_HEIGHTS, random_state=seed)}
    rows = []
    for name in ORDER:
        if KIND[name] == "projector":
            proj = projectors[name].fit(F_tr["wect"])
            tr, te = proj.transform(F_tr["wect"]), proj.transform(F_te["wect"])
        else:
            tr, te = F_tr[name], F_te[name]
        svm_acc, knn_acc = evaluate(tr, te, y_tr, y_te)
        rows.append({"method": name, "seed": seed, "dim": tr.shape[1], "svm": svm_acc, "knn": knn_acc})
    return rows


df = pd.DataFrame([row for seed in range(N_SEEDS) for row in run_seed(seed)])
agg = (df.groupby("method")
       .agg(svm_mean=("svm", "mean"), svm_std=("svm", "std"),
            knn_mean=("knn", "mean"), knn_std=("knn", "std"), dim=("dim", "first"))
       .reindex(ORDER))
agg["role"] = [ROLE[m] for m in ORDER]
df.to_csv(C.figure_dir(FOLDER) / "figure2a-dist-classifiers.csv", index=False)
print(agg.round(3).to_string())

            svm_mean  svm_std  knn_mean  knn_std   dim         role
method                                                             
mass           0.198    0.038     0.202    0.024     1      control
pixels         0.688    0.043     0.750    0.032  4096     baseline
wect           0.654    0.030     0.639    0.046  3072     baseline
wect_pca       0.669    0.038     0.640    0.046    29  dim control
wect_rand      0.594    0.053     0.572    0.062    48  dim control
diwect         0.987    0.007     0.998    0.003    48    invariant
wect_align     1.000    0.000     1.000    0.000  3072       oracle


## Figure

In [4]:
fig, ax = plt.subplots(figsize=(8.0, 3.8))
x = np.arange(len(ORDER))
for off, col, fill in [(-0.19, "svm", "#b8bcc2"), (0.19, "knn", "white")]:   # SVM filled, 1-NN open
    samples = [df.loc[df.method == m, col].to_numpy() for m in ORDER]
    C.box_plot(ax, samples, x + off, [fill] * len(ORDER), width=0.32)
    C.median_labels(ax, samples, x + off)
C.chance_line(ax, N_CLASSES)
ax.set_xticks(x); ax.set_xticklabels([TICKS.get(m, m) for m in ORDER])
ax.set_ylim(0, 1.08); ax.set_ylabel("test accuracy")
handles = [plt.Rectangle((0, 0), 1, 1, facecolor=f, edgecolor="#222222", lw=0.8) for f in ("#b8bcc2", "white")]
ax.legend(handles, ["Gaussian kernel SVM", "1-NN"], ncol=2, loc="upper left", fontsize=8)
C.save_pdf(fig, FOLDER, "figure2a-dist-classifiers")

saved -> figures/2-dist-classifiers-synthetic/figure2a-dist-classifiers.pdf
